# Метрики классификации — простыми словами

*(бинарная + многоклассовая)*

Учебный ноутбук: как понять, **насколько хорошо** модель отвечает «да/нет» (болен/здоров, спам/не спам) и **«какой из нескольких классов»** (кошка/собака/попугай).

Разбираем **матрицу ошибок**, **Accuracy, Precision, Recall, Specificity, F1, ROC AUC, Log Loss, PR AUC** — и для **нескольких классов** режимы `average`: **macro / weighted / micro**.

**Что понадобится**
- Python + Jupyter / VS Code / Colab
- `numpy`, `matplotlib`, `scikit-learn`

Запускайте ячейки **по порядку**.

---

## План

1. [Словарь: TP, TN, FP, FN](#dict)
2. [Матрица ошибок (confusion matrix)](#cm)
3. [Accuracy — доля правильных ответов](#acc)
4. [Precision — доверие к «да»](#prec)
5. [Recall — полнота «да»](#rec)
6. [Specificity — как хорошо узнаём «нет»](#spec)
7. [F1-score — баланс Precision и Recall](#f1)
8. [Порог вероятности и trade-off](#threshold)
9. [ROC AUC — качество разделения классов](#roc)
10. [Log Loss — качество вероятностей](#logloss)
11. [PR AUC — когда положительный класс редкий](#pr)
12. [Многоклассовая классификация: те же метрики, другой `average`](#multiclass)
13. [Шпаргалка и мини-практика](#итог)


<a id="dict"></a>
## 1. Словарь (обязательно)

| Термин | Простыми словами |
|--------|------------------|
| **Бинарная классификация** | Два класса: обычно **0** (отрицательный) и **1** (положительный) |
| **Положительный класс (1)** | То, что «ищем»: болезнь, спам, мошенничество |
| **Отрицательный класс (0)** | «Норма»: здоров, не спам, честная операция |
| **`predict`** | Жёсткий ответ 0/1 (после порога) |
| **`predict_proba`** | Вероятности; для класса 1 берут `[:, 1]` |
| **TP** (True Positive) | Сказали «да», и правда **да** |
| **TN** (True Negative) | Сказали «нет», и правда **нет** |
| **FP** (False Positive) | Сказали «да», а правда **нет** (ложная тревога) |
| **FN** (False Negative) | Сказали «нет», а правда **да** (пропуск) |
| **Порог (threshold)** | Если $p \ge t$ → класс 1, иначе 0 (часто $t=0.5$) |
| **Дисбаланс** | Классов сильно **неравное** число (990 здоровых, 10 больных) |

### Картинка «четыре клетки»

```text
                    Правда: ДА (1)     Правда: НЕТ (0)
Модель: ДА (1)         TP                  FP
Модель: НЕТ (0)        FN                  TN
```

> **FP** — «ложно запаниковали».  
> **FN** — «пропустили важное».

Цена FP и FN в жизни **разная** — поэтому одной Accuracy мало.


<a id="cm"></a>
## 2. Матрица ошибок — основа всех метрик

Возьмём **10 пациентов** (учебный пример).  
Класс **1 = болен**, **0 = здоров**.

| Пациент | Правда $y$ | Предсказание $\hat{y}$ | Тип |
|---------|------------|-------------------------|-----|
| 1 | 1 | 1 | TP |
| 2 | 1 | 1 | TP |
| 3 | 1 | 1 | TP |
| 4 | 1 | 0 | **FN** (пропустили) |
| 5 | 0 | 0 | TN |
| 6 | 0 | 0 | TN |
| 7 | 0 | 0 | TN |
| 8 | 0 | 0 | TN |
| 9 | 0 | 0 | TN |
| 10 | 0 | 1 | **FP** (ложная тревога) |

Итого: **TP=3, TN=5, FP=1, FN=1** (всего 10).

Дальше почти все формулы — просто **доли из этих четырёх чисел**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# Наш «ручной» пример: 10 пациентов
y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 1, 0, 0, 0, 0, 0, 0, 1])

cm = confusion_matrix(y_true, y_pred)
# В sklearn: строки = правда, столбцы = предсказание
# [[TN, FP],
#  [FN, TP]]  если метки [0, 1]
print("confusion_matrix (строки=правда, столбцы=прогноз):")
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(cm, display_labels=["здоров (0)", "болен (1)"]).plot(ax=ax, colorbar=False)
ax.set_title("Матрица ошибок — 10 пациентов")
plt.tight_layout()
plt.show()


<a id="acc"></a>
## 3. Accuracy — доля правильных ответов

### Вопрос

> Какой **процент** объектов модель классифицировала **правильно**?

Из 100 писем правильно 92 → Accuracy = **92%**.

### Формула

$$
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
$$

На 10 пациентах:

$$
Accuracy = \frac{3 + 5}{10} = 0.8 = 80\%
$$

### Когда применять

- классы **примерно сбалансированы**;
- цена FP и FN **примерно одинакова**.

### Главный минус: дисбаланс

990 здоровых, 10 больных.  
Модель **всегда** говорит «здоров»:

$$
Accuracy = 990/1000 = 99\%
$$

Выглядит «отлично», но **ни одного** больного не нашли.  
Поэтому после Accuracy почти всегда смотрят **Precision / Recall / F1 / PR AUC**.

### Главное

> Accuracy — процент **всех** правильных ответов.  
> Обманчива при **дисбалансе**.


In [ ]:
acc_hand = (tp + tn) / (tp + tn + fp + fn)
acc_sk = accuracy_score(y_true, y_pred)
print(f"Accuracy вручную: {acc_hand:.2f}")
print(f"Accuracy sklearn: {acc_sk:.2f}")

# --- Ловушка дисбаланса ---
y_imb = np.array([0] * 990 + [1] * 10)
y_always0 = np.zeros_like(y_imb)  # всегда «здоров»
print(f"\nAlways-0 на дисбалансе: accuracy = {accuracy_score(y_imb, y_always0):.3f}")
print(f"  recall (больных нашли): {recall_score(y_imb, y_always0, zero_division=0):.3f}")
print("→ 99% accuracy, но 0% recall по больным — модель бесполезна для поиска болезни.")


<a id="prec"></a>
## 4. Precision — точность положительных предсказаний

### Вопрос

> Из всех, кого модель назвала **положительными**, сколько **действительно** положительные?

Модель сказала «болен» 100 раз, реально больны 80 → Precision = **80%**.

### Формула

$$
Precision = \frac{TP}{TP + FP}
$$

На 10 пациентах: $TP=3$, $FP=1$ → $3/(3+1) = 0.75$.

### Физический смысл

> Если модель сказала **«да»**, насколько ей можно **доверять**?

Спам-фильтр: Precision = «какая доля писем в папке Спам — настоящий спам?»

### Когда важен

**FP очень дороги:**

1. тяжёлый диагноз «наугад»;  
2. блокировка карты;  
3. дорогое обследование;  
4. пометка «спам» (важные письма пропадут).

Лучше **реже** говорить «да», но когда сказали — быть правыми.

### Ловушка

Модель нашла **2** больных, оба верны → Precision = **100%**,  
но из 100 больных пропустила **98**. Precision высокий, польза мала.

### Главное

> Precision: *«Сказали „да“ — как часто правы?»*  
> Боится **FP**.


In [ ]:
prec_hand = tp / (tp + fp)
prec_sk = precision_score(y_true, y_pred)
print(f"Precision вручную: {prec_hand:.2f}")
print(f"Precision sklearn: {prec_sk:.2f}")

# Высокий precision, низкий recall
y_t = np.array([1] * 100 + [0] * 100)
y_p = np.zeros_like(y_t)
y_p[0] = 1  # нашли ровно одного больного, и он правда болен
y_p[1] = 1  # второго тоже
print("\nМодель нашла только 2 больных (оба верны):")
print(f"  precision={precision_score(y_t, y_p):.2f}, recall={recall_score(y_t, y_p):.2f}")


<a id="rec"></a>
## 5. Recall — полнота (Sensitivity, TPR)

### Вопрос

> Из всех **реально** положительных, сколько модель **нашла**?

Реально больны 100, нашли 90 → Recall = **90%**.

### Формула

$$
Recall = \frac{TP}{TP + FN}
$$

На 10 пациентах: $3/(3+1) = 0.75$.

Синонимы: **Sensitivity**, **TPR** (True Positive Rate).

### Физический смысл

> Если объект **действительно** «да», какова вероятность, что модель его **обнаружит**?

### Когда важен

**FN очень опасны:**

1. рак / тяжёлая болезнь;  
2. мошенничество;  
3. пожар;  
4. брак на производстве.

Лучше **лишний раз** поднять тревогу, чем пропустить.

### Ловушка

Модель говорит **всем** «болен» → Recall = **100%**, но куча FP — модель почти бесполезна.

### Precision vs Recall (запомнить)

| | Precision | Recall |
|--|-----------|--------|
| Вопрос | Сказали «да» — правы? | Реально «да» — нашли? |
| Боится | **FP** | **FN** |

Их почти всегда смотрят **вместе**, затем **F1**.


In [ ]:
rec_hand = tp / (tp + fn)
rec_sk = recall_score(y_true, y_pred)
print(f"Recall вручную: {rec_hand:.2f}")
print(f"Recall sklearn: {rec_sk:.2f}")

# Всегда «болен»
y_t2 = np.array([1] * 100 + [0] * 900)
y_all1 = np.ones_like(y_t2)
print("\nВсегда предсказываем 1:")
print(f"  recall={recall_score(y_t2, y_all1):.2f}, precision={precision_score(y_t2, y_all1):.2f}, "
      f"accuracy={accuracy_score(y_t2, y_all1):.2f}")


<a id="spec"></a>
## 6. Specificity — специфичность (TNR)

### Вопрос

> Из всех **реально отрицательных**, сколько правильно назвали отрицательными?

Реально здоровы 100, правильно 95 → Specificity = **95%**.

### Формула

$$
Specificity = \frac{TN}{TN + FP}
$$

На 10 пациентах: $TN=5$, $FP=1$ → $5/6 \approx 0.833$.

Также: **TNR** (True Negative Rate).

Связь с FPR:

$$
FPR = \frac{FP}{FP + TN} = 1 - Specificity
$$

### Медицинская пара

| Метрика | Вопрос |
|---------|--------|
| **Recall (чувствительность)** | Сколько **больных** нашли? |
| **Specificity (специфичность)** | Сколько **здоровых** правильно признали здоровыми? |

### Когда важна

FP дороги: ложный тяжёлый диагноз, блокировка карты, лишние обследования.

### Главное

> Specificity: *«Если объект действительно „нет“, поймёт ли модель?»*  
> Зеркало к Recall: Recall ↓ FN, Specificity ↓ FP.

В sklearn **отдельной** `specificity_score` нет — считают из `confusion_matrix`.


In [ ]:
spec_hand = tn / (tn + fp)
fpr_hand = fp / (fp + tn)
print(f"Specificity = {spec_hand:.3f}")
print(f"FPR         = {fpr_hand:.3f}")
print(f"1 - spec    = {1 - spec_hand:.3f}  (должно совпасть с FPR)")

# Через sklearn
tn2, fp2, fn2, tp2 = confusion_matrix(y_true, y_pred).ravel()
specificity = tn2 / (tn2 + fp2)
print(f"Specificity (из CM): {specificity:.3f}")


<a id="f1"></a>
## 7. F1-score — гармония Precision и Recall

### Вопрос

> Насколько хорошо модель **одновременно** находит положительных **и** не спамит ложными тревогами?

### Формула (гармоническое среднее)

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

Пример: Precision = 0.8, Recall = 0.6:

$$
F1 = 2 \cdot \frac{0.8 \cdot 0.6}{0.8 + 0.6} = 2 \cdot \frac{0.48}{1.4} \approx 0.686
$$

Не обычное среднее: если одна метрика **очень** низкая, F1 **сильнее** падает.

### Когда применять

- дисбаланс классов;  
- важны **и** Precision, **и** Recall;  
- Accuracy обманывает.

Мошенничество, болезни, спам, дефекты.

### Минусы

- **не** учитывает TN напрямую;  
- иногда всё равно смотрят Precision и Recall **отдельно** (разные цены ошибок).

### Главное

> Высокий F1 ≈ хороший **баланс** «нашли много» + «мало ложных тревог».


In [ ]:
p, r = precision_score(y_true, y_pred), recall_score(y_true, y_pred)
f1_hand = 2 * p * r / (p + r)
f1_sk = f1_score(y_true, y_pred)
print(f"Precision={p:.3f}, Recall={r:.3f}")
print(f"F1 вручную: {f1_hand:.3f}")
print(f"F1 sklearn: {f1_sk:.3f}")

print("\nclassification_report:")
print(classification_report(y_true, y_pred, target_names=["здоров", "болен"]))


<a id="threshold"></a>
## 8. Порог вероятности: Precision ↔ Recall

Большинство моделей сначала дают **вероятность**, не класс.

| Пациент | $p$ (болен) | Истина |
|---------|-------------|--------|
| A | 0.95 | 1 |
| B | 0.90 | 1 |
| C | 0.80 | 0 |
| D | 0.60 | 1 |
| E | 0.40 | 0 |
| F | 0.20 | 0 |

**Порог 0.5:** $p \ge 0.5$ → болен.

| | Прогноз |
|--|---------|
| A,B,C,D | 1 |
| E,F | 0 |

- Правда больные: A,B,D → все найдены → **TP=3, FN=0**, Recall=1  
- Здоровый C → FP=1; E,F → TN=2  
- FPR = $1/(1+2) \approx 0.33$  
- Precision = $3/(3+1)=0.75$

Если поднять порог до **0.85** — меньше FP (C отвалится), но можно потерять D (FN↑, Recall↓).

> **Низкий порог** → больше «да» → выше Recall, ниже Precision.  
> **Высокий порог** → реже «да» → выше Precision, ниже Recall.

ROC/PR AUC как раз смотрят **все** пороги сразу.


In [ ]:
# Таблица из текста
patients = ["A", "B", "C", "D", "E", "F"]
y = np.array([1, 1, 0, 1, 0, 0])
prob = np.array([0.95, 0.90, 0.80, 0.60, 0.40, 0.20])

print(f"{'порог':>6} | {'P':>5} | {'R':>5} | {'F1':>5} | {'FPR':>5} | предсказания")
print("-" * 55)
for thr in [0.9, 0.7, 0.5, 0.3]:
    pred = (prob >= thr).astype(int)
    # аккуратно, если нет положительных предсказаний
    p = precision_score(y, pred, zero_division=0)
    r = recall_score(y, pred, zero_division=0)
    f1 = f1_score(y, pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) else 0
    print(f"{thr:6.2f} | {p:5.2f} | {r:5.2f} | {f1:5.2f} | {fpr:5.2f} | {pred.tolist()}")


<a id="roc"></a>
## 9. ROC AUC — площадь под ROC-кривой

### Вопрос

> Насколько хорошо модель **отделяет** положительный класс от отрицательного **при всех порогах**?

Не «сколько правильных при 0.5», а «умеет ли **ранжировать**: больным ставить score выше, чем здоровым?»

### Как строится ROC

Для каждого порога:

- **TPR (Recall)** = $TP/(TP+FN)$ — ось **Y**  
- **FPR** = $FP/(FP+TN)$ — ось **X**

Соединяем точки → **ROC-кривая**.  
**AUC** = площадь под ней (обычно от ~0.5 до 1).

### Интерпретация AUC

| AUC | Смысл (грубо) |
|-----|----------------|
| 1.0 | идеал |
| 0.9 | очень хорошо |
| 0.8 | хорошо |
| 0.7 | средне |
| 0.5 | как подброс монетки |

**Вероятностный смысл:**  
AUC ≈ вероятность, что **случайный больной** получит **больший** score, чем **случайный здоровый**.  
AUC=0.92 → в 92% таких пар ranking верный.

### Когда применять

- есть **вероятности** / scores;  
- порог ещё **не зафиксирован**;  
- сравнение моделей.

### Минусы

- при **сильном дисбалансе** может быть **слишком оптимистичным** (много TN → FPR легко мал);  
- не говорит, **какой** порог брать;  
- не показывает число FP/FN при выбранном пороге.

### Важно в коде

```python
y_prob = model.predict_proba(X_test)[:, 1]  # вероятности класса 1
auc = roc_auc_score(y_test, y_prob)
```

Используют **`predict_proba`**, не `predict`!  
`[:, 1]` — столбец вероятности **положительного** класса.


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, RocCurveDisplay

# Маленький пример с вероятностями (пациенты A–F)
auc_small = roc_auc_score(y, prob)
print(f"ROC AUC на 6 пациентах: {auc_small:.3f}")

fpr, tpr, thr = roc_curve(y, prob)
print("Точки ROC (FPR, TPR, threshold):")
for a, b, t in zip(fpr, tpr, thr):
    print(f"  FPR={a:.2f}, TPR={b:.2f}, thr={t:.2f}")

fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc_small).plot(ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="случайный AUC=0.5")
ax.legend(loc="lower right")
ax.set_title("ROC-кривая (мини-пример)")
plt.tight_layout()
plt.show()


In [ ]:
# Более реалистичный синтетический пример
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y_cls = make_classification(
    n_samples=800, n_features=10, n_informative=5,
    weights=[0.7, 0.3], flip_y=0.05, random_state=42
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_cls, test_size=0.3, random_state=0, stratify=y_cls)

clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
y_hat = clf.predict(X_te)
y_prob = clf.predict_proba(X_te)[:, 1]

print("На test (порог 0.5 по умолчанию):")
print(f"  Accuracy  = {accuracy_score(y_te, y_hat):.3f}")
print(f"  Precision = {precision_score(y_te, y_hat):.3f}")
print(f"  Recall    = {recall_score(y_te, y_hat):.3f}")
print(f"  F1        = {f1_score(y_te, y_hat):.3f}")
print(f"  ROC AUC   = {roc_auc_score(y_te, y_prob):.3f}  ← по вероятностям")

fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_predictions(y_te, y_prob, ax=ax)
ax.set_title("ROC AUC — логистическая регрессия")
plt.tight_layout()
plt.show()


<a id="logloss"></a>
## 10. Log Loss — логарифмическая потеря

**Часто используется как loss** (в т.ч. под капотом логистической регрессии).

### Вопрос

> Насколько хорошо модель оценивает **вероятности**, а не только итоговый 0/1?

### Пример

Человек **болен** ($y=1$):

| Модель | $p$ | Класс при 0.5 | Accuracy | Интуиция |
|--------|-----|---------------|----------|----------|
| A | 0.51 | болен | верно | «еле-еле» |
| B | 0.99 | болен | верно | почти уверена |

Accuracy одинаковая. **Log Loss** скажет: B гораздо лучше.

Если $y=1$, а $p=0.01$ (уверена, что здоров) — **огромный** штраф.

### Формула (один объект)

$$
-(y \log p + (1-y)\log(1-p))
$$

По выборке — **среднее**. Чем **меньше**, тем лучше. Идеал → 0.

### Когда применять

- сравнивают **вероятностные** модели;  
- обучение logreg / нейросетей;  
- важно качество **калибровки/уверенности**.

### Минусы

- менее «бизнесовый», чем Accuracy/F1;  
- одна **уверенная** ошибка сильно портит среднее.

### Главное

> Accuracy/F1 смотрят на **класс**.  
> Log Loss смотрит на **вероятность** и сильно бьёт за уверенный промах.


In [ ]:
from sklearn.metrics import log_loss

# y=1, разные уверенности
for p in [0.99, 0.51, 0.01]:
    # log_loss ждёт список/массив; labels=[0,1] для одной точки
    ll = log_loss([1], [p], labels=[0, 1])
    print(f"y=1, p={p:.2f} → log_loss ≈ {ll:.3f}")

print()
# На нашем clf
ll = log_loss(y_te, y_prob)
print(f"Log Loss модели на test: {ll:.3f}")

# «Испорченные» вероятности: перепутаем и сделаем сверхуверенными ошибки
y_bad_prob = 1 - y_prob  # инвертировали
print(f"Log Loss инвертированных prob: {log_loss(y_te, y_bad_prob):.3f}  (хуже)")


<a id="pr"></a>
## 11. PR AUC — площадь под Precision–Recall

### Вопрос

> Насколько хорошо при **всех порогах** модель держит баланс **Precision ↔ Recall**?

Аналог ROC AUC, но оси:

| | ROC | PR |
|--|-----|-----|
| X | FPR | **Recall** |
| Y | TPR (Recall) | **Precision** |

**PR AUC** (часто через `average_precision_score`) — площадь под PR-кривой.  
Ближе к **1** — лучше.

### Когда особенно нужна

Положительный класс **очень редкий**:

- 100 мошенничеств из 100 000;  
- 1 больной из 1000.

### Почему ROC AUC может обмануть

10 000 объектов: 9990 отриц., 10 положит.  
Ошиблись на 20 здоровых → FPR = $20/9990 \approx 0$ — «красиво».  
ROC AUC выглядит отлично, а **поиск редкого класса** может быть слабым.  
**Precision** при этом падает → **PR AUC** это покажет.

### Ассоциация

Полицейский ищет преступников:

- **Recall** — сколько преступников нашёл;  
- **Precision** — сколько из задержанных — реально преступники;  
- **PR AUC** — насколько этот баланс держится при разной «строгости».

### Главное

> При **сильном дисбалансе** для редкого positive чаще смотрят **PR AUC + Precision/Recall/F1**,  
> а не только ROC AUC.


In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, PrecisionRecallDisplay

# Сильный дисбаланс
X_r, y_r = make_classification(
    n_samples=3000, n_features=12, n_informative=6,
    weights=[0.97, 0.03], flip_y=0.01, random_state=7
)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_r, y_r, test_size=0.3, random_state=0, stratify=y_r
)
clf_r = LogisticRegression(max_iter=2000).fit(Xr_tr, yr_tr)
prob_r = clf_r.predict_proba(Xr_te)[:, 1]
pred_r = clf_r.predict(Xr_te)

print(f"Доля positive в test: {yr_te.mean():.3f}")
print(f"Accuracy  = {accuracy_score(yr_te, pred_r):.3f}  ← легко выглядит высокой")
print(f"Precision = {precision_score(yr_te, pred_r, zero_division=0):.3f}")
print(f"Recall    = {recall_score(yr_te, pred_r, zero_division=0):.3f}")
print(f"F1        = {f1_score(yr_te, pred_r, zero_division=0):.3f}")
print(f"ROC AUC   = {roc_auc_score(yr_te, prob_r):.3f}")
print(f"PR AUC    = {average_precision_score(yr_te, prob_r):.3f}  (average precision)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(yr_te, prob_r, ax=axes[0])
axes[0].set_title("ROC (дисбаланс)")
PrecisionRecallDisplay.from_predictions(yr_te, prob_r, ax=axes[1])
axes[1].set_title("Precision–Recall (дисбаланс)")
plt.tight_layout()
plt.show()

print("\nBaseline PR (случайный уровень ≈ доля positive):", f"{yr_te.mean():.3f}")
print("PR AUC нужно сравнивать с этой «случайной» планкой, не с 0.5 как у ROC.")


<a id="multiclass"></a>
## 12. Многоклассовая классификация — тех же метрик почти достаточно

### Новых метрик почти нет

В **небинарной** (многоклассовой) классификации используют **те же**:

1. **Accuracy**  
2. **Precision**  
3. **Recall**  
4. **F1-score**

Меняется не «название метрики», а **способ собрать одно число из нескольких классов**.

### Проблема

Раньше был один «положительный» класс.  
Теперь, например, **три**:

| Класс |
|-------|
| Кошка |
| Собака |
| Попугай |

**Как посчитать Precision?** Для кого он «положительный»?

### Идея sklearn: one-vs-rest по очереди

Для **каждого** класса считают метрику так, будто задача **бинарная**:

```text
Кошка   против  «не кошка»
Собака  против  «не собака»
Попугай против  «не попугай»
```

Получаются **три** Precision (и три Recall, три F1…).  
Дальше их нужно **объединить** в одно число.

Для этого в `precision_score` / `recall_score` / `f1_score` есть параметр **`average`**.

---

### 1. `average="binary"`

Только для **бинарной** классификации (как в начале ноутбука).  
В многоклассовой **не применяется** (sklearn выдаст ошибку).

---

### 2. `average="macro"` — простое среднее по классам

1. Считаем метрику **для каждого** класса.  
2. Берём **обычное** среднее (все классы с **равным** весом).

Пример Precision:

| Класс | Precision |
|-------|-----------|
| Кошка | 0.90 |
| Собака | 0.80 |
| Попугай | 0.70 |

$$
\text{Macro Precision} = \frac{0.9 + 0.8 + 0.7}{3} = 0.8
$$

**Особенность:** даже если кошек **10 000**, а попугаев **10**, macro говорит:  
«классы **одинаково** важны».

**Когда:** каждый класс важен **сам по себе** (редко ошибаться на любом, в т.ч. на малом).

---

### 3. `average="weighted"` — среднее с весами по размеру класса

То же «сначала по классам», но вес класса ≈ **число объектов** этого класса (support).

Объектов: кошка 10 000, собака 1 000, попугай 10 → **кошки тянут** итог сильнее.

**Когда:** классы **несбалансированы** — на практике **очень часто**.

---

### 4. `average="micro"` — сначала все TP/FP/FN вместе

**Не** усредняет Precision классов.  
Складывает **все** TP, **все** FP, **все** FN по схеме one-vs-rest (глобально) и **потом** считает метрику — как одну большую «суммарную» задачу.

**Когда:** каждый **объект** одинаково важен, независимо от класса.

> Замечание: для multi-class **micro-F1 = micro-Precision = micro-Recall = Accuracy**  
> (все они сводятся к доле правильно угаданных меток). Удобно как проверка «всё ли понимаю».

---

### Как ведут себя режимы на дисбалансе (интуиция)

10 000 кошек, 10 попугаев.  
Модель **идеально** знает кошек, **плохо** — попугаев.

| Режим | Ожидание | Почему |
|-------|----------|--------|
| **macro** | **низкий** | попугай «весит» как кошка |
| **weighted** | **высокий** | почти все объекты — кошки |
| **micro** | **почти высокий** | считает по **объектам** в сумме |

Запомните треугольник:

1. **Macro** — одинаково любит **все классы**  
2. **Weighted** — любит **большие** классы  
3. **Micro** — смотрит на **все объекты** вместе  

---

### Что выбрать?

| Цель | `average` |
|------|-----------|
| Каждый класс одинаково важен | **`macro`** |
| Классы несбалансированы (частый выбор на практике) | **`weighted`** |
| Одна общая метрика «по всем объектам» | **`micro`** |
| Ровно 2 класса | **`binary`** (или опустить — default) |

### Accuracy

**Accuracy** в многоклассовой задаче — по-прежнему:

> доля объектов, у которых предсказанный класс **совпал** с истинным.

Ей **не** нужен `average` в том же смысле.  
Но при дисбалансе accuracy снова может **врать** — как в бинарном случае.

### Вызов

```python
from sklearn.metrics import precision_score, recall_score, f1_score

precision_score(y_test, y_pred, average="macro")      # или "weighted" / "micro"
recall_score(y_test, y_pred, average="weighted")
f1_score(y_test, y_pred, average="micro")
```

Полезно также:

```python
# метрика ПО КАЖДОМУ классу (без усреднения)
precision_score(y_test, y_pred, average=None)
classification_report(y_test, y_pred)
```

### Самое главное

> Новых метрик почти нет.  
> Есть **те же** Accuracy / Precision / Recall / F1 и вопрос:  
> **как усреднить по классам** — `macro` / `weighted` / `micro`.

Ниже — игрушечный пример «кошка / собака / попугай» и сравнение режимов.


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

# ----- 1) Ручной смысл macro: три Precision → среднее -----
# Допустим, one-vs-rest Precision уже посчитаны «по классам»
per_class_precision = np.array([0.90, 0.80, 0.70])  # кошка, собака, попугай
macro_p = per_class_precision.mean()
print("Пример из текста:")
print(f"  Precision по классам: {per_class_precision}")
print(f"  Macro Precision = {macro_p:.2f}")
print()

# ----- 2) Игрушечные y_true / y_pred: 3 класса -----
# 0=кошка, 1=собака, 2=попугай
# Нарочно: кошек много и они почти идеальны; попугаев мало и путаем
y_true_m = np.array(
    [0] * 20 + [1] * 8 + [2] * 4  # 20 кошек, 8 собак, 4 попугая
)
y_pred_m = np.array(
    [0] * 19 + [1]      # 19/20 кошек верно, 1 кошку назвали собакой
    + [1] * 6 + [0] + [2]  # 6 собак верно, 1→кошка, 1→попугай
    + [2, 2, 0, 1]         # 2 попугая верно, 1→кошка, 1→собака
)

labels = [0, 1, 2]
names = ["кошка", "собака", "попугай"]

print("Матрица ошибок (строки=правда, столбцы=прогноз):")
print(confusion_matrix(y_true_m, y_pred_m, labels=labels))
print()

# По классам (average=None)
p_each = precision_score(y_true_m, y_pred_m, labels=labels, average=None, zero_division=0)
r_each = recall_score(y_true_m, y_pred_m, labels=labels, average=None, zero_division=0)
f_each = f1_score(y_true_m, y_pred_m, labels=labels, average=None, zero_division=0)
support = np.bincount(y_true_m)

print(f"{'класс':10s} | {'P':>6} | {'R':>6} | {'F1':>6} | support")
print("-" * 48)
for name, p, r, f, s in zip(names, p_each, r_each, f_each, support):
    print(f"{name:10s} | {p:6.3f} | {r:6.3f} | {f:6.3f} | {s:3d}")
print()

# Разные average
for avg in ["macro", "weighted", "micro"]:
    print(
        f"average={avg!r:12}: "
        f"P={precision_score(y_true_m, y_pred_m, average=avg, zero_division=0):.3f}  "
        f"R={recall_score(y_true_m, y_pred_m, average=avg, zero_division=0):.3f}  "
        f"F1={f1_score(y_true_m, y_pred_m, average=avg, zero_division=0):.3f}"
    )

acc = accuracy_score(y_true_m, y_pred_m)
print(f"\nAccuracy = {acc:.3f}")
print(
    "Проверка: micro-F1 == Accuracy? →",
    np.isclose(f1_score(y_true_m, y_pred_m, average="micro"), acc),
)

print("\nclassification_report (там уже есть macro / weighted):")
print(classification_report(y_true_m, y_pred_m, target_names=names, digits=3, zero_division=0))

fig, ax = plt.subplots(figsize=(4.8, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_true_m, y_pred_m, labels=labels),
    display_labels=names,
).plot(ax=ax, colorbar=False)
ax.set_title("Многоклассовая матрица ошибок")
plt.tight_layout()
plt.show()

# ----- 3) Weighted «на пальцах» для Precision -----
# weighted = sum(precision_c * support_c) / sum(support)
weighted_p_hand = np.sum(p_each * support) / support.sum()
weighted_p_sk = precision_score(y_true_m, y_pred_m, average="weighted", zero_division=0)
print(f"Weighted Precision вручную: {weighted_p_hand:.3f}")
print(f"Weighted Precision sklearn: {weighted_p_sk:.3f}")
print()
print("Сравните macro vs weighted: если редкий класс слабый, macro обычно ниже.")


<a id="итог"></a>
## 13. Самое главное + шпаргалка

### Все метрики одной таблицей

| Метрика | Вопрос | Формула / идея | sklearn |
|---------|--------|----------------|---------|
| **Accuracy** | % всех правильных? | $(TP+TN)/N$ | `accuracy_score` |
| **Precision** | Среди предсказанных «да» — сколько правда? | $TP/(TP+FP)$ | `precision_score` |
| **Recall** | Среди реальных «да» — сколько нашли? | $TP/(TP+FN)$ | `recall_score` |
| **Specificity** | Среди реальных «нет» — сколько угадали? | $TN/(TN+FP)$ | из `confusion_matrix` |
| **F1** | Баланс P и R? | гарм. среднее | `f1_score` |
| **ROC AUC** | Хорошо ли ранжирует при всех порогах? | площадь TPR–FPR | `roc_auc_score` (**proba**) |
| **PR AUC** | Баланс P–R при всех порогах? | площадь PR | `average_precision_score` |
| **Log Loss** | Хороши ли вероятности? | $-\mathbb{E}[\log p_y]$ ↓ | `log_loss` (**proba**) |


### Многоклассовость (`average`)

| `average` | Смысл | Когда |
|-----------|--------|--------|
| **`binary`** | один positive | только 2 класса |
| **`macro`** | среднее по классам, веса равны | каждый класс важен |
| **`weighted`** | среднее с весом = support | дисбаланс (часто) |
| **`micro`** | глобальные TP/FP/FN | важны объекты; micro-F1 = accuracy |

### Что выбрать?

| Ситуация | Ориентир |
|----------|----------|
| Классы сбалансированы, ошибки равноценны | Accuracy (+ F1) |
| FP дороги (ложная тревога) | Precision, Specificity |
| FN дороги (пропуск) | Recall |
| Нужен один score при дисбалансе | **F1** |
| Сравниваем модели, порог не выбран | **ROC AUC** |
| Редкий positive (мошенничество, болезнь) | **PR AUC**, Precision/Recall |
| Важна калибровка вероятностей | **Log Loss** |
| Всегда полезно | **Confusion matrix** глазами |
| 3+ класса | те же P/R/F1 + **`average`** (`macro`/`weighted`/`micro`) |
| Редкий важный класс среди многих | смотрите **per-class** + **macro**, не только weighted |

### Train / Valid / Test

Как и в регрессии / валидации:

- крутим порог и выбираем модель по **valid / CV**;  
- **test** — один раз;  
- не гонитесь только за train-метриками.

### Мини-глоссарий

| Слово | Смысл |
|-------|--------|
| **TPR** | = Recall |
| **FPR** | = 1 − Specificity |
| **Threshold** | отсечка вероятности |
| **Ranking** | умение ставить больным score выше |
| **Average precision** | практическая оценка PR AUC в sklearn |
| **macro / weighted / micro** | как усреднять P/R/F1 по классам |
| **support** | сколько объектов класса (вес в weighted) |


### Мини-практика

1. Обучите `LogisticRegression` на `make_classification` (можно с лёгким дисбалансом).  
2. На test выведите: matrix, accuracy, precision, recall, f1, roc_auc, pr_auc, log_loss.  
3. Переберите пороги 0.3 / 0.5 / 0.7 и сравните Precision/Recall/F1.  
4. Коротко напишите (в `print`): *в вашей задаче что дороже — FP или FN?*

5. **(Многокласс)** Возьмите 3+ класса (`make_classification(..., n_classes=3)`),  
   сравните `f1_score(..., average="macro")` и `average="weighted"`.  
   Кратко: *какой режим ниже и почему?*


In [ ]:
# ===== Мини-практика / эталонный шаблон =====
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
)

np.random.seed(0)
X, y = make_classification(
    n_samples=1000, n_features=8, n_informative=5,
    weights=[0.8, 0.2], random_state=0
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Confusion matrix [[TN FP],[FN TP]]:")
print(confusion_matrix(y_test, y_pred))
print()
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred):.3f}")
print(f"F1       : {f1_score(y_test, y_pred):.3f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_prob):.3f}")
print(f"PR AUC   : {average_precision_score(y_test, y_prob):.3f}")
print(f"Log Loss : {log_loss(y_test, y_prob):.3f}")

print("\nПороги:")
for thr in [0.3, 0.5, 0.7]:
    pred = (y_prob >= thr).astype(int)
    print(f"  thr={thr:.1f}: P={precision_score(y_test, pred, zero_division=0):.3f} "
          f"R={recall_score(y_test, pred, zero_division=0):.3f} "
          f"F1={f1_score(y_test, pred, zero_division=0):.3f}")

print("\nЕсли FN дороже (пропуск болезни) — снижайте порог, растите Recall.")
print("Если FP дороже (ложная блокировка) — повышайте порог, растите Precision.")


## Что делать дальше

1. Прогоните ноутбук и сверьте ручные TP/TN/FP/FN со `sklearn`.  
2. Свяжите с **«Валидация…»**: метрики на CV, test — финал.  
3. В реальной задаче **сначала** решите, что дороже — FP или FN — и **потом** выбирайте метрику и порог.  
4. При редком positive не верьте одной Accuracy / иногда и одному ROC AUC — смотрите **PR-кривую**.
5. В многоклассовой задаче явно пишите `average=...` и смотрите `classification_report` по классам.

### Главная мысль занятия

> Классификация — это не только «сколько угадали»,  
> а **какие ошибки** мы делаем и **насколько уверены**.  
> Матрица ошибок + Precision/Recall + (ROC или PR) AUC — базовый набор для **бинарной** задачи.
> Для **многих классов** добавьте ясный `average` (macro/weighted/micro) и отчёт **по классам**.

Удачи!
